# 🌿 Plant Disease Detection — YOLOv8-OBB + TensorFlow Classifier

**Environment:** `tf_env` — Python 3.10 | NumPy 1.26.4 | TF 2.10.1 | PyTorch 2.7.1+cu118

**Pipeline:**
1. Convert DOTA labels → YOLOv8-OBB normalized format
2. Verify labels
3. Train YOLOv8m-OBB (7 severity classes) on GPU
4. Evaluate OBB model
5. Two-stage inference: OBB detector → TF CustomCNN classifier
6. Batch inference + CSV export

**Models:**
- OBB Detector : YOLOv8m-OBB — 7 classes (Blight_a/b/c, Danger_a/b, Healthy_a/b)
- Classifier   : CustomCNN .keras — 38 disease classes — 99.28% val accuracy

## 0. Environment Verification

In [ ]:
import sys
import numpy as np
import tensorflow as tf
import torch
import cv2
from ultralytics import YOLO

print('=' * 55)
print('  Environment')
print('=' * 55)
print(f'  Python      : {sys.version.split()[0]}')
print(f'  NumPy       : {np.__version__}')
print(f'  TensorFlow  : {tf.__version__}')
print(f'  PyTorch     : {torch.__version__}')
print(f'  OpenCV      : {cv2.__version__}')
print()
print('  GPU Status')
print(f'  TF  GPU : {tf.config.list_physical_devices("GPU")}')
print(f'  PT  GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU"}')
print(f'  VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB' if torch.cuda.is_available() else '')
print('=' * 55)
print('  ✅ Ready to train!')

## 1. Configuration

In [ ]:
import os

# ── Paths ──────────────────────────────────────────────────────────────────
OBB_ROOT        = r'D:\leaf.v1-yolo_leaf1.yolov5-obb'
CLASSIFIER_PATH = r'D:\Leaf dataset 2\outputs\CustomCNN\CustomCNN_best.keras'
OUTPUT_DIR      = os.path.join(OBB_ROOT, 'outputs')
DATA_YAML       = os.path.join(OBB_ROOT, 'data.yaml')

# ── OBB Classes ────────────────────────────────────────────────────────────
OBB_CLASSES = [
    'Blight_a', 'Blight_b', 'Blight_c',
    'Danger_a', 'Danger_b',
    'Healthy_a', 'Healthy_b'
]
CLASS_MAP = {name: idx for idx, name in enumerate(OBB_CLASSES)}

# ── Classifier Classes (38) ────────────────────────────────────────────────
CLASSIFIER_CLASSES = [
    'Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust',
    'Apple___healthy', 'Blueberry___healthy',
    'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy',
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_',
    'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy',
    'Grape___Black_rot', 'Grape___Esca_(Black_Measles)',
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy',
    'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot',
    'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy',
    'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy',
    'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew',
    'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot',
    'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold',
    'Tomato___Septoria_leaf_spot',
    'Tomato___Spider_mites Two-spotted_spider_mite', 'Tomato___Target_Spot',
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus', 'Tomato___Tomato_mosaic_virus',
    'Tomato___healthy'
]

# ── Severity Display Map ───────────────────────────────────────────────────
SEVERITY_MAP = {
    'Healthy_a': '🟢 Healthy (Confirmed)',
    'Healthy_b': '🟡 Healthy (Minor marks)',
    'Danger_a' : '🟠 Warning — Early Stage',
    'Danger_b' : '🟠 Caution — Moderate',
    'Blight_a' : '🔴 Blight — Mild',
    'Blight_b' : '🔴 Blight — Moderate',
    'Blight_c' : '🔴 Blight — SEVERE ⚠️',
}

SEVERITY_COLORS = {
    'Healthy_a': (0,   200, 0),
    'Healthy_b': (0,   180, 80),
    'Danger_a' : (255, 165, 0),
    'Danger_b' : (255, 120, 0),
    'Blight_a' : (220, 50,  50),
    'Blight_b' : (180, 20,  20),
    'Blight_c' : (130, 0,   0),
}

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('✅ Configuration loaded')
print(f'   OBB root     : {OBB_ROOT}')
print(f'   Classifier   : {CLASSIFIER_PATH}')
print(f'   Output dir   : {OUTPUT_DIR}')
print(f'   OBB classes  : {OBB_CLASSES}')
print(f'   Cls classes  : {len(CLASSIFIER_CLASSES)}')

## 2. Convert DOTA Labels → YOLOv8-OBB Format

**DOTA format** (absolute pixels):  
`x1 y1 x2 y2 x3 y3 x4 y4 ClassName Difficult`

**YOLOv8-OBB format** (normalized 0-1):  
`class_idx x1 y1 x2 y2 x3 y3 x4 y4`

> Skip this cell if you already ran the conversion previously.

In [ ]:
import glob
from PIL import Image
from tqdm import tqdm

def convert_dota_to_yoloobb(label_dir, image_dir, output_dir, class_map):
    os.makedirs(output_dir, exist_ok=True)
    label_files = glob.glob(os.path.join(label_dir, '*.txt'))
    converted, skipped, empty = 0, 0, 0

    for lf in tqdm(label_files, desc=f'Converting {os.path.basename(os.path.dirname(label_dir))}'):
        filename = os.path.basename(lf)
        stem     = os.path.splitext(filename)[0]

        # Find matching image
        img_path = None
        for ext in ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']:
            candidate = os.path.join(image_dir, stem + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break

        if img_path is None:
            skipped += 1
            open(os.path.join(output_dir, filename), 'w').close()
            continue

        W, H = Image.open(img_path).size

        out_lines = []
        with open(lf, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) < 9:
                    continue
                try:
                    x1,y1 = float(parts[0]), float(parts[1])
                    x2,y2 = float(parts[2]), float(parts[3])
                    x3,y3 = float(parts[4]), float(parts[5])
                    x4,y4 = float(parts[6]), float(parts[7])
                    cls    = parts[8]
                except ValueError:
                    continue

                if cls not in class_map:
                    continue

                def nx(v): return max(0.0, min(1.0, v / W))
                def ny(v): return max(0.0, min(1.0, v / H))

                out_lines.append(
                    f"{class_map[cls]} "
                    f"{nx(x1):.6f} {ny(y1):.6f} "
                    f"{nx(x2):.6f} {ny(y2):.6f} "
                    f"{nx(x3):.6f} {ny(y3):.6f} "
                    f"{nx(x4):.6f} {ny(y4):.6f}"
                )

        with open(os.path.join(output_dir, filename), 'w') as f:
            f.write('\n'.join(out_lines))

        if len(out_lines) == 0:
            empty += 1
        converted += 1

    print(f'  ✅ Converted: {converted} | ⚠️ No image: {skipped} | 📄 Background: {empty}')


for split in ['train', 'valid', 'test']:
    print(f'\n🔄 {split}...')
    convert_dota_to_yoloobb(
        label_dir  = os.path.join(OBB_ROOT, split, 'labelTxt'),
        image_dir  = os.path.join(OBB_ROOT, split, 'images'),
        output_dir = os.path.join(OBB_ROOT, split, 'labels'),
        class_map  = CLASS_MAP,
    )

print('\n✅ All labels converted!')

## 3. Verify Labels

In [ ]:
print('=' * 60)
print('  Label Verification')
print('=' * 60)

for split in ['train', 'valid', 'test']:
    label_files  = glob.glob(os.path.join(OBB_ROOT, split, 'labels', '*.txt'))
    non_empty    = sum(1 for f in label_files if os.path.getsize(f) > 0)
    total_boxes  = 0
    class_counts = {i: 0 for i in range(len(OBB_CLASSES))}

    for lf in label_files:
        with open(lf) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 9:
                    total_boxes += 1
                    class_counts[int(parts[0])] += 1

    print(f'\n📁 {split}:')
    print(f'   Files      : {len(label_files)}')
    print(f'   Non-empty  : {non_empty}')
    print(f'   Total boxes: {total_boxes}')
    max_count = max(class_counts.values()) if class_counts.values() else 1
    for idx, name in enumerate(OBB_CLASSES):
        bar = '█' * int(class_counts[idx] / max_count * 25)
        print(f'   {name:<12} {class_counts[idx]:>5}  {bar}')

    sample = next((f for f in label_files if os.path.getsize(f) > 0), None)
    if sample:
        with open(sample) as f:
            print(f'   Sample: {f.readline().strip()}')

## 4. Create data.yaml

In [ ]:
yaml_content = f"""# YOLOv8-OBB Plant Disease Dataset
path: {OBB_ROOT}
train: train/images
val:   valid/images
test:  test/images

nc: {len(OBB_CLASSES)}
names: {OBB_CLASSES}
"""

with open(DATA_YAML, 'w') as f:
    f.write(yaml_content)

print('✅ data.yaml saved:')
print('-' * 40)
print(yaml_content)

## 5. Train YOLOv8m-OBB on GPU

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8m-obb.pt')  # pretrained backbone

results = model.train(
    data     = DATA_YAML,
    epochs   = 100,
    imgsz    = 640,
    batch    = 16,          # reduce to 8 if VRAM runs out
    device   = 0,           # RTX 4060
    project  = OUTPUT_DIR,
    name     = 'yolov8m_obb',
    patience = 20,          # early stopping

    # Optimizer
    optimizer    = 'AdamW',
    lr0          = 0.001,
    cos_lr       = True,
    weight_decay = 0.0005,

    # Loss weights
    cls = 1.0,   # boosted for class imbalance (Healthy only 2.9%)
    box = 7.5,
    dfl = 1.5,

    # Augmentation
    fliplr     = 0.5,
    flipud     = 0.2,
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    mosaic     = 1.0,
    mixup      = 0.1,       # helps rare Healthy_a/b classes
    copy_paste = 0.1,       # copies rare instances into images

    # Misc
    amp     = True,         # mixed precision — faster on RTX 4060
    plots   = True,
    verbose = True,
)

print(f'\n✅ Training complete!')
print(f'   Best mAP50   : {results.results_dict.get("metrics/mAP50(B)", 0):.4f}')
print(f'   Best mAP50-95: {results.results_dict.get("metrics/mAP50-95(B)", 0):.4f}')

## 6. Evaluate on Validation & Test Sets

In [ ]:
# Find best weights
weight_candidates = glob.glob(os.path.join(OUTPUT_DIR, '**', 'best.pt'), recursive=True)
best_weights = sorted(weight_candidates)[-1] if weight_candidates else None
print(f'📍 Best weights: {best_weights}')

trained_model = YOLO(best_weights)

for split in ['val', 'test']:
    metrics = trained_model.val(
        data    = DATA_YAML,
        split   = split,
        imgsz   = 640,
        device  = 0,
        verbose = False,
    )
    print(f'\n📊 {split.upper()} Results:')
    print(f'   mAP50     : {metrics.box.map50:.4f}')
    print(f'   mAP50-95  : {metrics.box.map:.4f}')
    print(f'   Precision : {metrics.box.mp:.4f}')
    print(f'   Recall    : {metrics.box.mr:.4f}')

## 7. Per-Class AP Chart

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

val_metrics = trained_model.val(
    data    = DATA_YAML,
    split   = 'val',
    imgsz   = 640,
    device  = 0,
    verbose = True,
)

rows = []
for i, name in enumerate(OBB_CLASSES):
    rows.append({
        'Class'   : name,
        'AP50'    : float(val_metrics.box.ap50[i]) if i < len(val_metrics.box.ap50) else 0.0,
        'AP50-95' : float(val_metrics.box.ap[i])   if i < len(val_metrics.box.ap)   else 0.0,
    })

df = pd.DataFrame(rows).sort_values('AP50', ascending=False)
print(df.to_string(index=False))

colors = ['#e74c3c' if 'Blight'  in c else
          '#e67e22' if 'Danger'  in c else
          '#2ecc71' for c in df['Class']]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(df['Class'], df['AP50'], color=colors, edgecolor='white', linewidth=0.5)
ax.axhline(val_metrics.box.map50, color='navy', linestyle='--', alpha=0.7,
           label=f'mAP50 = {val_metrics.box.map50:.3f}')
ax.set_ylim(0, 1.05)
ax.set_ylabel('AP@50', fontsize=12)
ax.set_title('Per-Class AP@50 — YOLOv8m-OBB', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Blight'),
    mpatches.Patch(color='#e67e22', label='Danger'),
    mpatches.Patch(color='#2ecc71', label='Healthy'),
    plt.Line2D([0],[0], color='navy', linestyle='--',
               label=f'mAP50 = {val_metrics.box.map50:.3f}'),
])
plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, 'per_class_ap50.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Saved → {plot_path}')

## 8. Load Both Models for Inference

In [ ]:
import tensorflow as tf
import numpy as np
import cv2

# ── Load OBB detector ─────────────────────────────────────────────────────
print('Loading OBB detector...')
obb_model = YOLO(best_weights)
print(f'  ✅ OBB model loaded: {best_weights}')

# ── Load TF Classifier ────────────────────────────────────────────────────
print('Loading TF classifier...')

# Enable GPU memory growth to avoid OOM with YOLO also using GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'  GPU memory growth enabled for {len(gpus)} GPU(s)')

classifier = tf.keras.models.load_model(CLASSIFIER_PATH)
print(f'  ✅ Classifier loaded: {CLASSIFIER_PATH}')
print(f'  Input shape : {classifier.input_shape}')
print(f'  Output shape: {classifier.output_shape}')

# Warmup pass
_ = classifier.predict(np.zeros((1, 224, 224, 3), dtype='float32'), verbose=0)
print('  ✅ Warmup done — both models ready!')

## 9. Two-Stage Inference Pipeline

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def extract_crop(image, obb_box, padding=10):
    """Extract axis-aligned bounding crop from an OBB detection."""
    pts = obb_box.xyxyxyxy[0].cpu().numpy().astype(int)
    x1  = max(0, pts[:, 0].min() - padding)
    y1  = max(0, pts[:, 1].min() - padding)
    x2  = min(image.shape[1], pts[:, 0].max() + padding)
    y2  = min(image.shape[0], pts[:, 1].max() + padding)
    return image[y1:y2, x1:x2], (x1, y1, x2, y2)


def classify_crop(crop_bgr):
    """Run TF CustomCNN classifier on a BGR crop."""
    crop_rgb     = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    crop_resized = cv2.resize(crop_rgb, (224, 224))
    inp          = np.expand_dims(crop_resized.astype('float32') / 255.0, axis=0)
    preds        = classifier.predict(inp, verbose=0)
    idx          = np.argmax(preds[0])
    return CLASSIFIER_CLASSES[idx], float(preds[0][idx])


def predict_image(image_path, conf_threshold=0.3, show=True, save_path=None):
    """
    Full two-stage pipeline:
      Stage 1: YOLOv8-OBB  → detect lesion regions + severity grade
      Stage 2: TF CustomCNN → classify specific disease from each crop
    """
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f'Image not found: {image_path}')

    # Stage 1 — OBB detection
    obb_results = obb_model(image_path, conf=conf_threshold, verbose=False)[0]
    detections  = []

    if obb_results.obb is None or len(obb_results.obb) == 0:
        print('⚠️  No lesions detected.')
        return detections

    for i, box in enumerate(obb_results.obb):
        severity_idx  = int(box.cls[0])
        severity_name = OBB_CLASSES[severity_idx]
        obb_conf      = float(box.conf[0])

        crop, (x1, y1, x2, y2) = extract_crop(image, box)
        if crop.size == 0:
            continue

        # Stage 2 — classify
        disease_name, cls_conf = classify_crop(crop)

        detections.append({
            'region'      : i + 1,
            'disease'     : disease_name.replace('___', ' — '),
            'severity'    : SEVERITY_MAP[severity_name],
            'severity_key': severity_name,
            'det_conf'    : obb_conf,
            'cls_conf'    : cls_conf,
            'bbox'        : (x1, y1, x2, y2),
        })

    # ── Visualisation ─────────────────────────────────────────────────────
    if show or save_path:
        img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(figsize=(12, 8))
        ax.imshow(img_rgb)

        for det in detections:
            x1, y1, x2, y2 = det['bbox']
            color = tuple(c / 255 for c in SEVERITY_COLORS[det['severity_key']])
            rect  = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=color, facecolor=(*color, 0.08)
            )
            ax.add_patch(rect)
            label = (
                f"#{det['region']} {det['disease'].split(' — ')[-1]}\n"
                f"{det['severity'].split(' ',1)[1]} "
                f"| det:{det['det_conf']:.0%} cls:{det['cls_conf']:.0%}"
            )
            ax.text(x1, max(y1-5, 10), label, color='white', fontsize=7,
                    bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85))

        ax.set_title(
            f"Detections: {len(detections)}  |  {os.path.basename(image_path)}",
            fontsize=12, fontweight='bold'
        )
        ax.axis('off')
        plt.tight_layout()

        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f'💾 Saved → {save_path}')
        if show:
            plt.show()
        plt.close()

    # ── Print summary ─────────────────────────────────────────────────────
    print(f'\n{"─"*65}')
    print(f'  📍 {os.path.basename(image_path)}')
    print(f'  🔍 {len(detections)} detection(s)')
    print(f'{"─"*65}')
    for det in detections:
        print(f"  Region {det['region']:>2}: {det['disease']}")
        print(f"           {det['severity']}")
        print(f"           Det: {det['det_conf']:.2%}  |  Cls: {det['cls_conf']:.2%}")
        print()

    return detections


print('✅ Pipeline ready — use predict_image(path) to run inference')

## 10. Run on Test Images

In [ ]:
test_images = []
for ext in ['*.jpg', '*.JPG', '*.jpeg', '*.png', '*.PNG']:
    test_images += glob.glob(os.path.join(OBB_ROOT, 'test', 'images', ext))
test_images = list(set(test_images))[:5]

if not test_images:
    print('⚠️  No test images found.')
else:
    print(f'Running on {len(test_images)} test images...\n')
    for img_path in test_images:
        save_path = os.path.join(
            OUTPUT_DIR,
            'pred_' + os.path.basename(img_path).replace(' ', '_')
        )
        predict_image(img_path, conf_threshold=0.3, show=True, save_path=save_path)

## 11. Custom Image Inference

In [ ]:
# ── Change this to any leaf image path ───────────────────────────────────
CUSTOM_IMAGE = r'D:\path\to\your\leaf_image.jpg'

if os.path.exists(CUSTOM_IMAGE):
    predict_image(
        CUSTOM_IMAGE,
        conf_threshold = 0.25,
        show           = True,
        save_path      = os.path.join(OUTPUT_DIR, 'custom_prediction.png'),
    )
else:
    print(f'⚠️  File not found: {CUSTOM_IMAGE}')
    print('    Update CUSTOM_IMAGE path and re-run.')

## 12. Batch Inference → CSV Export

In [ ]:
import pandas as pd
from datetime import datetime

def batch_predict(image_dir, conf_threshold=0.3, output_csv=None):
    paths = []
    for ext in ['*.jpg', '*.JPG', '*.jpeg', '*.png', '*.PNG']:
        paths += glob.glob(os.path.join(image_dir, ext))
    paths = list(set(paths))
    print(f'Found {len(paths)} images in {image_dir}')

    rows = []
    for img_path in tqdm(paths, desc='Batch inference'):
        try:
            dets = predict_image(img_path, conf_threshold=conf_threshold,
                                 show=False, save_path=None)
            if dets:
                for det in dets:
                    rows.append({
                        'image'         : os.path.basename(img_path),
                        'region'        : det['region'],
                        'disease'       : det['disease'],
                        'severity'      : det['severity'],
                        'det_confidence': round(det['det_conf'], 4),
                        'cls_confidence': round(det['cls_conf'], 4),
                    })
            else:
                rows.append({
                    'image': os.path.basename(img_path), 'region': 0,
                    'disease': 'No detection', 'severity': '—',
                    'det_confidence': 0, 'cls_confidence': 0,
                })
        except Exception as e:
            print(f'  ⚠️  {os.path.basename(img_path)}: {e}')

    df = pd.DataFrame(rows)
    if output_csv is None:
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_csv = os.path.join(OUTPUT_DIR, f'batch_predictions_{ts}.csv')

    df.to_csv(output_csv, index=False)
    print(f'\n✅ {len(df)} rows saved → {output_csv}')
    print('\n📊 Severity distribution:')
    print(df['severity'].value_counts().to_string())
    return df


# Run on test set
df_results = batch_predict(
    image_dir      = os.path.join(OBB_ROOT, 'test', 'images'),
    conf_threshold = 0.3,
)

## 13. Final Summary

In [ ]:
print('=' * 60)
print('  🌿 Plant Disease Detection — Summary')
print('=' * 60)
print()
print('  Environment')
print(f'    tf_env : TF 2.10.1 | PyTorch 2.7.1+cu118 | NumPy 1.26.4')
print(f'    GPU    : RTX 4060 Laptop (8GB VRAM)')
print()
print('  Stage 1 — YOLOv8m-OBB Detector')
print(f'    Classes : {OBB_CLASSES}')
print(f'    Weights : {best_weights}')
print(f'    Train   : 11,822 images | Valid: 261 | Test: 226')
print()
print('  Stage 2 — CustomCNN Classifier (TensorFlow)')
print(f'    Classes  : {len(CLASSIFIER_CLASSES)} disease classes')
print(f'    Val acc  : 99.28%')
print(f'    Weights  : {CLASSIFIER_PATH}')
print()
print('  Example Output')
print('    Tomato — Late_blight | 🔴 Blight — SEVERE ⚠️ | Det:87% Cls:94%')
print()
print(f'  All outputs → {OUTPUT_DIR}')
print('=' * 60)